# How good is the published evidence for QPOP?

QPOP tests ~155 drug combinations on a patient's tumour cells in the lab and predicts which combination will work.
Its papers report high accuracy. This notebook re-checks those numbers using only what the papers themselves publish.

**Real:** every count in sections 1-3 (sources in `docs/sources.md`).
**Simulated:** section 4, which asks how much of a reported accuracy can come from choosing the cutoff on the same small data set.

In [1]:
import math
import numpy as np
import plotly.graph_objects as go

def wilson(k, n, z=1.96):
    """95% Wilson interval for k successes out of n."""
    p = k / n
    c = (p + z*z/(2*n)) / (1 + z*z/n)
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return c - h, c + h

def fisher_two_sided(a, b, c, d):
    """Fisher exact test for the 2x2 table [[a, b], [c, d]]."""
    r1, c1, n = a + b, a + c, a + b + c + d
    pr = lambda x: math.comb(c1, x) * math.comb(n - c1, r1 - x) / math.comb(n, r1)
    p0 = pr(a)
    return sum(pr(x) for x in range(max(0, r1 + c1 - n), min(r1, c1) + 1) if pr(x) <= p0 * (1 + 1e-9))

## 1. AML (Sahib et al. 2024): from 63 samples to the headline number

Counts from the paper's results section. The 86.2% accuracy is computed on the last bar only.

In [2]:
funnel = [("samples collected", 63), ("QPOP runs", 59), ("passed quality check", 55),
          ("evaluated vs outcome", 38), ("used for headline accuracy", 29)]
fig = go.Figure(go.Bar(x=[n for _, n in funnel], y=[s for s, _ in funnel], orientation="h",
                       text=[n for _, n in funnel], textposition="outside"))
fig.update_layout(yaxis=dict(autorange="reversed"), xaxis_title="samples", height=330,
                  title="AML: 29 of 63 samples (46%) reach the accuracy figure")
fig

## 2. AML accuracy, re-computed with honest baselines

2x2 table from the paper: 15 true positives, 3 false negatives, 10 true negatives, 1 false positive.
Baseline: predict "responds" for everyone. Kappa measures agreement beyond chance (0 = chance, 1 = perfect).

In [3]:
TP, FN, TN, FP = 15, 3, 10, 1
N_AML = TP + FN + TN + FP
metrics = {"sensitivity": (TP, TP + FN), "specificity": (TN, TN + FP), "PPV": (TP, TP + FP),
           "NPV": (TN, TN + FN), "accuracy": (TP + TN, N_AML), "baseline: everyone responds": (TP + FN, N_AML)}
for name, (k, n) in metrics.items():
    lo, hi = wilson(k, n)
    print(f"{name:28s} {k:2d}/{n:2d} = {k/n:6.1%}   95% CI {lo:6.1%} - {hi:6.1%}")
po = (TP + TN) / N_AML
pe = ((TP + FP) * (TP + FN) + (TN + FN) * (TN + FP)) / N_AML**2
kappa = (po - pe) / (1 - pe)
p_fisher = fisher_two_sided(TP, FN, FP, TN)
print(f"Cohen's kappa = {kappa:.2f}, Fisher exact p = {p_fisher:.4f}")

sensitivity                  15/18 =  83.3%   95% CI  60.8% -  94.2%
specificity                  10/11 =  90.9%   95% CI  62.3% -  98.4%
PPV                          15/16 =  93.8%   95% CI  71.7% -  98.9%
NPV                          10/13 =  76.9%   95% CI  49.7% -  91.8%
accuracy                     25/29 =  86.2%   95% CI  69.4% -  94.5%
baseline: everyone responds  18/29 =  62.1%   95% CI  44.0% -  77.3%
Cohen's kappa = 0.72, Fisher exact p = 0.0003


In [4]:
names = list(metrics)
vals = [k / n for k, n in metrics.values()]
ci = [wilson(k, n) for k, n in metrics.values()]
fig = go.Figure(go.Bar(x=names, y=vals, error_y=dict(type="data", symmetric=False,
                       array=[h - v for v, (l, h) in zip(vals, ci)], arrayminus=[v - l for v, (l, h) in zip(vals, ci)])))
fig.update_layout(yaxis=dict(title="proportion", range=[0, 1.05]), height=420,
                  title=f"AML, n = {N_AML}: wide intervals; accuracy vs the do-nothing baseline")
fig

## 3. Lymphoma (NHL) numbers

2025 paper (Lee et al., full text read): the abstract puts "74.5% accuracy" next to "105 evaluable" cases, but the accuracy is computed on **47** cases
(page 9: "of the 47 patients evaluated ... QPOP accurately classified 18 and 17 patients"). 47 is 37% of the 126 QPOP cases.
The 2x2 table follows from the reported PPV 69.2% (18/26) and NPV 81.0% (17/21): TP 18, FP 8, TN 17, FN 4.
The cutoff was again chosen by ROC on the same patients. AUROC 0.69.

In [5]:
NHL = dict(TP=18, FP=8, TN=17, FN=4)
N_NHL = sum(NHL.values())
nhl = {"2022: complete response on QPOP-guided therapy": (5, 17),
       "2025: sensitivity": (NHL["TP"], NHL["TP"] + NHL["FN"]),
       "2025: specificity": (NHL["TN"], NHL["TN"] + NHL["FP"]),
       "2025: accuracy": (NHL["TP"] + NHL["TN"], N_NHL),
       "2025: baseline, everyone non-responder": (NHL["TN"] + NHL["FP"], N_NHL),
       "2025: ORR on QPOP-guided therapy": (16, 27),
       "AML 2024: accuracy": (TP + TN, N_AML)}
for name, (k, n) in nhl.items():
    lo, hi = wilson(k, n)
    print(f"{name:48s} {k:3d}/{n:3d} = {k/n:6.1%}   95% CI {lo:6.1%} - {hi:6.1%}")
po = (NHL["TP"] + NHL["TN"]) / N_NHL
pe = ((NHL["TP"] + NHL["FP"]) * (NHL["TP"] + NHL["FN"]) + (NHL["TN"] + NHL["FN"]) * (NHL["TN"] + NHL["FP"])) / N_NHL**2
print(f"2025 Cohen's kappa = {(po - pe) / (1 - pe):.2f}, Fisher p = {fisher_two_sided(NHL['TP'], NHL['FN'], NHL['FP'], NHL['TN']):.4f}")

2022: complete response on QPOP-guided therapy     5/ 17 =  29.4%   95% CI  13.3% -  53.1%
2025: sensitivity                                 18/ 22 =  81.8%   95% CI  61.5% -  92.7%
2025: specificity                                 17/ 25 =  68.0%   95% CI  48.4% -  82.8%
2025: accuracy                                    35/ 47 =  74.5%   95% CI  60.5% -  84.7%
2025: baseline, everyone non-responder            25/ 47 =  53.2%   95% CI  39.2% -  66.7%
2025: ORR on QPOP-guided therapy                  16/ 27 =  59.3%   95% CI  40.7% -  75.5%
AML 2024: accuracy                                25/ 29 =  86.2%   95% CI  69.4% -  94.5%
2025 Cohen's kappa = 0.49, Fisher p = 0.0010


## 4. Simulation: how much accuracy comes from tuning the cutoff on the same data?

In the AML paper the 0.65 cutoff was picked by ROC analysis on the same patients it was then scored on.
Here: two groups (responders, non-responders) whose lab scores differ by an effect size *d* (0 = the test carries no information).
For each simulated study we pick the best cutoff on the study's own patients, record the accuracy that study would report,
then score the same cutoff on 20,000 new patients. The gap is optimism. Response rate fixed at 62% as in the AML cohort.

In [6]:
rng = np.random.default_rng(0)
RESP = 18 / 29

def optimism(n, d, reps=2000):
    """Mean reported (same-data) accuracy and mean true accuracy for best-cutoff studies of size n."""
    y = rng.random((reps, n)) < RESP
    s = rng.normal(0, 1, (reps, n)) - d * y          # responders have lower viability scores
    order = np.argsort(s, axis=1)
    ys = np.take_along_axis(y, order, axis=1)
    # cutoff after position k: first k called responders
    tp = np.concatenate([np.zeros((reps, 1)), np.cumsum(ys, axis=1)], axis=1)
    fp = np.arange(n + 1)[None] - tp
    tn = (~y).sum(1, keepdims=True) - fp
    acc = (tp + tn) / n
    k = acc.argmax(1)
    reported = acc[np.arange(reps), k]
    srt = np.take_along_axis(s, order, axis=1)
    pad = np.concatenate([np.full((reps, 1), -np.inf), srt, np.full((reps, 1), np.inf)], axis=1)
    cut = (pad[np.arange(reps), k] + pad[np.arange(reps), k + 1]) / 2
    cut = np.where(np.isfinite(cut), cut, np.where(k == 0, -np.inf, np.inf))
    # true accuracy of each cutoff, exact from the normal distribution
    Phi = lambda x: 0.5 * (1 + np.vectorize(math.erf)(x / math.sqrt(2)))
    true = RESP * Phi(cut + d) + (1 - RESP) * (1 - Phi(cut))
    return reported.mean(), true.mean()

D = np.round(np.arange(0, 2.01, 0.25), 2)
NS = np.arange(10, 101, 10)
REP = np.zeros((len(NS), len(D))); TRUE = np.zeros_like(REP)
for i, n in enumerate(NS):
    for j, d in enumerate(D):
        REP[i, j], TRUE[i, j] = optimism(n, d)
j29 = [optimism(29, d) for d in D]
print("n = 29:  d   reported  true")
for d, (r, t) in zip(D, j29):
    print(f"      {d:4.2f}   {r:6.1%}  {t:6.1%}")

n = 29:  d   reported  true
      0.00    66.8%   56.1%
      0.25    69.3%   58.2%
      0.50    72.2%   61.0%
      0.75    74.8%   64.3%
      1.00    77.9%   68.4%
      1.25    81.3%   72.2%
      1.50    84.1%   75.9%
      1.75    86.7%   79.3%
      2.00    89.3%   82.5%


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=D, y=[r for r, _ in j29], mode="lines+markers", name="reported (cutoff tuned on same 29)"))
fig.add_trace(go.Scatter(x=D, y=[t for _, t in j29], mode="lines+markers", name="true, on new patients"))
fig.add_hline(y=RESP, line_dash="dot", annotation_text="everyone-responds baseline 62%")
fig.add_hline(y=25 / 29, line_dash="dash", annotation_text="AML paper 86.2%")
fig.update_layout(xaxis_title="effect size d (0 = test is useless)", yaxis_title="accuracy", height=430,
                  title="n = 29: what a tuned-cutoff study reports vs what it really delivers")
fig

### Optimism over study size and effect size (rotate it)

In [8]:
fig = go.Figure(go.Surface(x=D, y=NS, z=REP - TRUE, colorscale="Viridis",
                           colorbar=dict(title="optimism")))
fig.update_layout(scene=dict(xaxis_title="effect size d", yaxis_title="patients in study",
                             zaxis_title="reported minus true accuracy"),
                  title="Small studies with a tuned cutoff overstate accuracy", height=600)
fig

## 5. Lymphoma 2022 (Goh et al.): does QPOP-guided therapy beat the patient's previous treatment?

Real per-patient data from the supplement, Table S7: PFS on the treatment before QPOP and on the QPOP-guided treatment (17 patients).
The paper reports "10 of 17 patients (59%) had improved PFS" and calls the ratio "internally controlled".
Second check: Fig S2, patients with 2 or more prior lines, progressive disease at 6 months (QPOP 5 of 15, clinician-guided 11 of 18).

In [9]:
PFS_PREV = np.array([680, 284, 183, 96, 225, 35, 585, 97, 57, 137, 119, 68, 228, 32, 24, 80, 14])
PFS_QPOP = np.array([50, 49, 44, 32, 81, 18, 401, 105, 64, 217, 223, 129, 553, 81, 81, 709, 311])
RATIO = PFS_QPOP / PFS_PREV
n_up = int((RATIO > 1).sum()); n_p = len(RATIO)
sign_p = sum(math.comb(n_p, i) for i in range(n_p + 1) if math.comb(n_p, i) <= math.comb(n_p, n_up)) / 2**n_p
print(f"median PFS before: {np.median(PFS_PREV):.0f} days, on QPOP therapy: {np.median(PFS_QPOP):.0f} days")
print(f"ratio > 1 in {n_up}/{n_p}; sign test p = {sign_p:.2f}")
print(f"mean ratio {RATIO.mean():.2f} (largest single ratio {RATIO.max():.1f}); geometric mean {np.exp(np.log(RATIO).mean()):.2f}")
print(f"correlation of log(previous PFS) with log(ratio): {np.corrcoef(np.log(PFS_PREV), np.log(RATIO))[0, 1]:.2f}")
print(f"Fig S2 (>=2 prior lines): OR {5*7/(10*11):.2f}, Fisher p = {fisher_two_sided(5, 10, 11, 7):.3f}")

median PFS before: 97 days, on QPOP therapy: 81 days
ratio > 1 in 10/17; sign test p = 0.63
mean ratio 2.90 (largest single ratio 22.2); geometric mean 1.08
correlation of log(previous PFS) with log(ratio): -0.69
Fig S2 (>=2 prior lines): OR 0.32, Fisher p = 0.166


In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=PFS_PREV, y=PFS_QPOP, mode="markers", marker=dict(size=9), name="patient"))
fig.add_trace(go.Scatter(x=[10, 800], y=[10, 800], mode="lines", line=dict(dash="dot"), name="no change"))
fig.update_layout(xaxis=dict(type="log", title="PFS on previous treatment (days)"),
                  yaxis=dict(type="log", title="PFS on QPOP-guided treatment (days)"), height=450,
                  title="Points above the line improved. Short previous PFS -> large ratio almost automatically")
fig

**Reading it.** The median patient did worse on QPOP-guided therapy (81 vs 97 days), and neither the sign test nor the ratio average supports improvement.
The ratio is driven by how short the previous treatment lasted (correlation −0.69): heavily pre-treated patients whose last regimen failed fast
will often show a ratio above 1 on any next treatment. This is regression to the mean, not evidence for QPOP. In the wider ≥2-line group the paper's own Fig S2 is not significant (p = 0.17);
the significant OR 0.05 (p = 0.035) is from the narrower ≥3-line subgroup.

## 6. Lymphoma 2025 (Lee et al.): the same check on 29 patients, and what changed since 2022

Data supplement Table S2 lists PFS before and after QPOP-guided therapy for 29 patients; 17 of them appear in the 2022 paper too.
The paper reports that "approximately 59.3% of QPOP-guided patients had PFS ratios >1.3" (threshold from another study, its ref 14).

In [11]:
L25 = {"SLS-0607": (680, 50), "SLS-0922": (80, 773), "QPOP005": (24, 81), "2981-TY": (137, 217), "3707-YA": (97, 105),
       "QPOP016": (228, 553), "QPOP030": (57, 64), "QPOP032": (68, 129), "QPOP036": (14, 311), "QPOP044": (585, 474),
       "QPOP042": (96, 32), "3878-SK": (32, 81), "QPOP052": (119, 223), "QPOP059": (225, 81), "QPOP065": (35, 18),
       "QPOP069": (183, 44), "3834-FB": (37, 49), "4008-TB": (196, 725), "4184-AS": (95, 723), "4168-GC": (25, 55),
       "QPOP119": (485, 60), "4255-MM": (109, 681), "QPOP136": (161, 540), "4303-SA": (23, 33), "4241-SP": (165, 28),
       "QPOP174": (328, 267), "QPOP179": (251, 17), "QPOP180": (182, 202), "4123-HB": (54, 190)}
G22_IDS = ["SLS-0607", "3834-FB", "QPOP069", "QPOP042", "QPOP059", "QPOP065", "QPOP044", "3707-YA", "QPOP030",
           "2981-TY", "QPOP052", "QPOP032", "QPOP016", "3878-SK", "QPOP005", "SLS-0922", "QPOP036"]
for pid, a, b in zip(G22_IDS, PFS_PREV, PFS_QPOP):
    if L25[pid] != (a, b):
        print(f"{pid}: 2022 table ({a}, {b}) -> 2025 table {L25[pid]}")
P25 = np.array([v[0] for v in L25.values()]); Q25 = np.array([v[1] for v in L25.values()]); R25 = Q25 / P25
k = int((R25 > 1).sum()); n = len(R25)
sp = sum(math.comb(n, i) for i in range(n + 1) if math.comb(n, i) <= math.comb(n, k)) / 2**n
print(f"\n29 patients: median PFS before {np.median(P25):.0f} d, after {np.median(Q25):.0f} d; ratio > 1 in {k}/{n}, sign test p = {sp:.2f}")
print(f"ratio > 1.3: {(R25 > 1.3).sum()} patients; 16/27 = {16/27:.1%} (paper: 59.3%; 2 of 29 excluded from response evaluation)")
print(f"with the 2022 value for 3834-FB (ratio {49/284:.2f}) the count would be {(R25 > 1.3).sum() - 1} -> {15/27:.1%}")
print(f"correlation of log(previous PFS) with log(ratio): {np.corrcoef(np.log(P25), np.log(R25))[0, 1]:.2f}")

3834-FB: 2022 table (284, 49) -> 2025 table (37, 49)
QPOP044: 2022 table (585, 401) -> 2025 table (585, 474)
SLS-0922: 2022 table (80, 709) -> 2025 table (80, 773)

29 patients: median PFS before 109 d, after 105 d; ratio > 1 in 19/29, sign test p = 0.14
ratio > 1.3: 16 patients; 16/27 = 59.3% (paper: 59.3%; 2 of 29 excluded from response evaluation)
with the 2022 value for 3834-FB (ratio 0.17) the count would be 15 -> 55.6%
correlation of log(previous PFS) with log(ratio): -0.57


**Reading it.** Two of the three changed rows are longer follow-up on the QPOP-guided treatment, which is expected.
The third is not: patient 3834-FB's PFS on the *previous* treatment drops from 284 to 37 days. A finished treatment's PFS should not change,
and the change moves this patient from "worse" (ratio 0.17) to just above the 1.3 "improved" line (1.32). Neither paper explains it; it may be a correction or an error.
Over 29 patients the median PFS is about the same before and after (109 vs 105 days) and the sign test is not significant (p = 0.14).

## 7. Are the lab concentrations ones a patient can reach?

Cmax = peak blood concentration in patients. Values from Sahib 2024 (AML), supplement Tables S2-S3, which cite Liston & Davis, Clin Cancer Res 2017.
The AML paper says concentrations "were kept below 10% Cmax". The myeloma paper's ex vivo concentrations are from Rashid 2018 Table S12.
Files: `data/sahib2024_aml_tableS3_concentrations_uM.csv`, `data/rashid2018_myeloma_exvivo_tableS12_concentrations_uM.csv`.
AML Fig S3 also reports the concordance on all 38 evaluated runs: AUC 0.719 (22 responders, 16 non-responders), against 0.808 on the 29 runs kept for the headline.

In [12]:
import csv as _csv
aml = list(_csv.DictReader(open("../data/sahib2024_aml_tableS3_concentrations_uM.csv")))
mm = {r["drug"]: float(r["level_1"]) for r in _csv.DictReader(open("../data/rashid2018_myeloma_exvivo_tableS12_concentrations_uM.csv"))}
CMAX = {r["drug"]: float(r["cmax_uM"]) for r in aml}
CMAX["decitabine"] = 0.323   # Sahib 2024 Table S2 (Liston & Davis 2017)
aml_ratio = {r["drug"]: float(r["qpop_high_uM"]) / float(r["cmax_uM"]) for r in aml}
mm_ratio = {d: mm[d] / CMAX[d] for d in ("decitabine", "bortezomib", "panobinostat")}
print("AML drugs above the stated 10% of Cmax:", {d: f"{v:.0%}" for d, v in aml_ratio.items() if v > 0.10})
print("Myeloma 2018 ex vivo high dose / Cmax:", {d: f"{v:.3g}x" for d, v in mm_ratio.items()})

AML drugs above the stated 10% of Cmax: {'azacytidine': '20%', 'midostaurin': '25%', 'bortezomib': '20%', 'dasatinib': '20%', 'gilteritinib': '20%'}
Myeloma 2018 ex vivo high dose / Cmax: {'decitabine': '186x', 'bortezomib': '0.00641x', 'panobinostat': '0.0341x'}


In [13]:
fig = go.Figure()
fig.add_trace(go.Bar(x=list(aml_ratio), y=list(aml_ratio.values()), name="AML 2024 (high level)"))
fig.add_trace(go.Bar(x=[f"{d} (MM 2018)" for d in mm_ratio], y=list(mm_ratio.values()), name="Myeloma 2018 ex vivo (high level)"))
fig.add_hline(y=0.10, line_dash="dot", annotation_text="stated limit: 10% of Cmax")
fig.add_hline(y=1.0, line_dash="dash", annotation_text="Cmax")
fig.update_layout(yaxis=dict(type="log", title="lab concentration / patient Cmax"), height=480,
                  title="Myeloma: decitabine tested at ~190x what patients reach; bortezomib at 0.6%")
fig

**Reading it.** In AML, five drugs sit at about 20-25% of Cmax, twice the stated 10% limit. In the myeloma paper the comparison is lopsided:
decitabine, part of the "winning" Dec + MitoC pair, was tested at 93-186 times its patient Cmax, while bortezomib and panobinostat, the backbone of the clinical regimens it was compared with, were tested at 0.3-0.6% and 2-3% of theirs.
A drug given at a hundred times its reachable level against rivals at a fraction of theirs will tend to win the ranking. (Mitomycin C, melphalan and dexamethasone are not in the Cmax table, so they are not shown.)

## 8. Soft tissue sarcoma (Chan et al., NPJ Precis Oncol 2025, CC BY)

Reported (Results, Fig. 1): reports for 45 of 51 samples; concordance on **27 treatment outcomes from 14 patients**, from QPOP-guided therapy *or prior lines of treatment*;
responder = partial response **or stable disease** (PR 7, SD 9, PD 11); cutoff NCV 0.8305 chosen by ROC on the same outcomes (AUC 0.769, 0.585-0.954);
14 QPOP-responder vs 13 non-responder cases; OR 13.5; "total predictive value" 77.8%.
The 2x2 table is not printed, but these totals fix it uniquely: TP 12, FP 2, FN 4, TN 9, which gives exactly OR = 12x9 / (2x4) = 13.5.
File: `data/chan2025_sarcoma_concordance.csv`.

In [14]:
STS = dict(TP=12, FP=2, FN=4, TN=9)
n_sts = sum(STS.values())
assert STS["TP"] * STS["TN"] / (STS["FP"] * STS["FN"]) == 13.5 and STS["TP"] + STS["FP"] == 14 and STS["TP"] + STS["FN"] == 16
for name, (k, n) in {"accuracy": (STS["TP"] + STS["TN"], n_sts), "everyone responds (PR/SD) baseline": (16, n_sts),
                     "sensitivity": (STS["TP"], 16), "specificity": (STS["TN"], 11)}.items():
    lo, hi = wilson(k, n)
    print(f"{name:36s} {k:2d}/{n} = {k/n:6.1%}   95% CI {lo:6.1%} - {hi:6.1%}")
po = (STS["TP"] + STS["TN"]) / n_sts
pe = ((STS["TP"] + STS["FP"]) * 16 + (STS["TN"] + STS["FN"]) * 11) / n_sts**2
print(f"Cohen's kappa = {(po - pe) / (1 - pe):.2f}; Fisher p = {fisher_two_sided(STS['TP'], STS['FN'], STS['FP'], STS['TN']):.4f}"
      f" (treats 27 outcomes from 14 patients as independent)")

accuracy                             21/27 =  77.8%   95% CI  59.2% -  89.4%
everyone responds (PR/SD) baseline   16/27 =  59.3%   95% CI  40.7% -  75.5%
sensitivity                          12/16 =  75.0%   95% CI  50.5% -  89.8%
specificity                           9/11 =  81.8%   95% CI  52.3% -  94.9%
Cohen's kappa = 0.55; Fisher p = 0.0063 (treats 27 outcomes from 14 patients as independent)


**Reading it.** The association is real on paper (kappa about 0.5), but four things inflate it: about two outcomes per patient are counted as independent;
"stable disease" counts as a response; some outcomes are earlier treatments matched to cells sampled later; and the cutoff was tuned on the same 27 outcomes.
With 14 patients, the honest unit of evidence is small.

## What this says

1. **The AML signal is real, but small and selected.** Kappa 0.72 and Fisher p = 0.0003 mean QPOP did better than chance on these 29 samples.
   But the 29 are 46% of the samples collected. Deaths before treatment (8), loss to follow-up (4) and off-panel treatment (5) were removed, and those are not random exclusions.
2. **The "do nothing" baseline is 62%.** Calling every patient a responder scores 62.1%, so QPOP's 86.2% is +24 points, not +36 over a coin flip.
3. **The interval is wide.** The 95% CI for accuracy is 69%–95%. One more study of 29 patients could land anywhere in that range.
4. **The cutoff was tuned on the same patients.** In the simulation, a study of 29 patients with a tuned cutoff overstates accuracy by about 7–11 points at every effect size. A reported 86% is consistent with a true accuracy near 79%. The paper also used the first 22 samples to adjust drug doses before the 29 were scored, which adds more optimism, of a size that cannot be estimated from the paper.
5. **Reporting problems.** The AML AUC interval is given as 0.612–1.005, but an AUC cannot exceed 1. The NHL abstract places 74.5% beside 105 evaluable cases, but the accuracy uses 47 of 126 cases (37%).
6. **Lymphoma is weaker than AML.** On 47 cases, accuracy 74.5% against a 53% do-nothing baseline, AUROC 0.69. The authors themselves cite about 87% mean accuracy for other ex vivo assays. QPOP-guided vs clinician-guided comparisons are non-randomized (treatment allocation "nonrandomized", Methods).
7. **Lymphoma survival benefit is not shown.** Paired PFS (Goh 2022, Table S7): median 81 days on QPOP therapy vs 97 before, sign test p = 0.63; the PFS ratio reflects regression to the mean. The 2025 update (29 patients) shows the same: median 109 vs 105 days, sign test p = 0.14; and one patient's previous-treatment PFS changed between papers (284 to 37 days) with no explanation.
8. **No independent replication.** Every QPOP paper found comes from one group, with a company (KYAN Technologies) providing the analysis software.
9. **Lab doses.** AML: five drugs at ~20% of Cmax despite a stated 10% limit; on all 38 evaluated runs the AUC is 0.72, not 0.81. Myeloma: decitabine tested at 93-186x patient Cmax, bortezomib at under 1% of it.
10. **Sarcoma:** 77.8% accuracy on 27 outcomes from only 14 patients, against a 59% baseline, with stable disease counted as response and the cutoff tuned on the same data.
11. **Myeloma has no clinical QPOP data.** The 2018 myeloma paper is cell lines, mice and ex vivo samples, with no patient outcomes.

Bottom line: QPOP is a plausible lab test with early, single-group, optimistic evidence in blood cancers. It is not yet evidence that QPOP-guided treatment beats physician choice. None of the papers in `docs/sources.md` is a randomized comparison.